# 🔀 LangGraph Conditional Routing: The LLM Router Pattern

### Overview & Architectural Goals
In agentic workflows, an LLM frequently serves as an **intelligent router** deciding control flow:
- If the user's prompt requires calculation or external computation, the model issues a **tool call**, routing execution to a dedicated tool execution node.
- If the user's prompt is conversational or self-contained, the model produces a direct text response, routing straight to `END`.

This lab explores:
1. Binding callable tools to **`ChatGroq` (`llama-3.3-70b-versatile`)**.
2. Using the prebuilt **`ToolNode`** to execute function calls automatically.
3. Employing **`tools_condition`** as a conditional edge to dynamically branch execution based on whether tool calls exist.

## 📐 System Architecture: Router Flowchart

The diagram below illustrates the decision flow of the LLM Router Pattern:

<div align="center">
  <img src="images/06_langgraph_router_pattern.png" alt="LangGraph LLM Router Pattern Architecture" width="100%" />
</div>

<br/>

<details>
<summary><b>🔍 View Raw Mermaid Diagram Syntax</b></summary>

```mermaid
flowchart TD
    START([🚀 START]) --> LLM["🤖 tool_calling_llm<br/>ChatGroq: llama-3.3-70b-versatile"]
    LLM --> Cond{"🔀 tools_condition<br/>Inspects latest AIMessage"}
    Cond -->|Has tool_calls| Tools["🛠️ ToolNode<br/>Executes multiply(a, b)"]
    Cond -->|No tool_calls / Direct Answer| END([🏁 END])
    Tools --> END
```

</details>


## 1. Installation

Install `langchain_groq`, `langgraph`, and core packages.

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_groq langchain_core langgraph langgraph-prebuilt python-dotenv

## 2. Environment Setup (Groq API Key)

We check and configure `GROQ_API_KEY` from your environment or `.env` file.

In [ ]:
import os, getpass
from dotenv import load_dotenv
load_dotenv()

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("GROQ_API_KEY")
print("✅ GROQ_API_KEY configured successfully.")

## 3. Defining Tools & Binding to Groq LLM

We define a mathematical tool (`multiply`). LangChain inspects docstrings and type annotations to generate the JSON tool specification. We then bind it to Groq's flagship **`llama-3.3-70b-versatile`** model via `.bind_tools()`.

In [ ]:
from langchain_groq import ChatGroq

def multiply(a: int, b: int) -> int:
    """Multiply two integers together.

    Args:
        a: First integer factor.
        b: Second integer factor.
    """
    return a * b

# Initialize Groq chat model with deterministic temperature
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm_with_tools = llm.bind_tools([multiply])
print("✅ Tool 'multiply' bound to ChatGroq model.")

## 4. Graph Construction: `ToolNode` & `tools_condition`

- **`MessagesState`**: Standard state schema with a `messages` key using the `add_messages` reducer.
- **`ToolNode`**: Prebuilt node that extracts pending `tool_calls` from the last `AIMessage`, executes the Python function, and returns `ToolMessage` instances.
- **`tools_condition`**: Built-in conditional edge that inspects the latest message:
  - If it contains `tool_calls` -> routes to `"tools"`.
  - If no tool call was made -> routes to `END`.

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

# 1. Define the LLM node
def tool_calling_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# 2. Assemble StateGraph
builder = StateGraph(MessagesState)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode([multiply]))

# 3. Edges: START -> LLM -> tools_condition
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,
)
builder.add_edge("tools", END)

# 4. Compile the router graph
graph = builder.compile()
print("✅ Router graph compiled successfully.")

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Mermaid rendering skipped: {e}")

## 5. Execution Protocol: Sequence Diagram

<div align="center">
  <img src="images/seq_router_dispatch.png" alt="Conditional Routing Dispatch Sequence Diagram" width="100%" />
</div>

<br/>

<details>
<summary><b>🔍 View Raw Mermaid Diagram Syntax</b></summary>

```mermaid
sequenceDiagram
    autonumber
    actor User
    participant Engine as LangGraph Engine
    participant LLM as ChatGroq (llama-3.3-70b-versatile)
    participant Condition as tools_condition
    participant ToolNode as ToolNode([multiply])

    User->>Engine: graph.invoke({'messages': ['What is 2 multiplied by 2?']})
    Engine->>LLM: Invoke tool_calling_llm
    LLM-->>Engine: AIMessage(tool_calls=[multiply(a=2, b=2)])
    Engine->>Condition: Evaluate last message
    Condition-->>Engine: Path: 'tools'
    Engine->>ToolNode: Execute multiply(2, 2)
    ToolNode-->>Engine: ToolMessage(content='4')
    Engine-->>User: Final MessagesState
```
</details>

## 6. Test Case 1: Tool-Triggering Query (Routes to `ToolNode`)

We supply a prompt requesting multiplication. The router invokes `multiply` and outputs the resulting `ToolMessage`.

In [ ]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="Hello, what is 2 multiplied by 2?")]
result = graph.invoke({"messages": messages})

print("Execution Trace for Math Query:")
for m in result['messages']:
    m.pretty_print()

## 7. Test Case 2: Conversational Query (Routes directly to `END`)

When the input does not require computation, `tools_condition` detects that no tool call was generated and routes directly to `END` without invoking `ToolNode`.

In [ ]:
conversational_messages = [HumanMessage(content="Hello! What is your purpose?")]
direct_result = graph.invoke({"messages": conversational_messages})

print("Execution Trace for Conversational Query (Direct Response):")
for m in direct_result['messages']:
    m.pretty_print()